# 10 - Model Comparison

Aggregates every model's results from `all_metrics.json` (written by notebooks 03-09) into
one table and the headline charts. Run last.

## Setup — load metrics

In [1]:
import sys
sys.path.insert(0, "../src")
import json
import pandas as pd
import plotly.express as px
from pathlib import Path
from hybrid_recsys.config import ARTIFACTS_METRICS

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1200, height=600, scale=2)
    except Exception:
        pass
    fig.show()

metrics = json.loads((ARTIFACTS_METRICS / "all_metrics.json").read_text())
print(f"{len(metrics)} models in all_metrics.json")

8 models in all_metrics.json


## Results table

In [2]:
rows = []
for label, mm in metrics.items():
    rows.append({"Model": label, "RMSE": mm["rmse"], "MAE": mm["mae"],
                 "P@10": mm["k10"]["precision"], "R@10": mm["k10"]["recall"], "F1@10": mm["k10"]["f1"]})
results = pd.DataFrame(rows).set_index("Model")
display(
    results.style
    .highlight_min(subset=["RMSE", "MAE"], color="#d4edda")
    .highlight_max(subset=["F1@10"], color="#d4edda")
    .format("{:.4f}")
)

,RMSE,MAE,P@10,R@10,F1@10
Model,,,,,
Global Mean,1.0609,0.8339,0.0661,0.0888,0.0758
Popularity,2.7120,2.4856,0.4791,0.8356,0.6090
Content-Based,1.0462,0.7831,0.1828,0.2394,0.2073
User-Based k-NN,1.0401,0.8119,0.1029,0.1500,0.1221
Item-Based k-NN,0.8336,0.6223,0.3537,0.5579,0.4329
SVD,0.8108,0.6080,0.2901,0.4482,0.3523
Weighted Hybrid,0.8095,0.6074,0.2917,0.4505,0.3541
Stacked Hybrid,0.8054,0.6029,0.3432,0.5334,0.4176


## RMSE & MAE by model

In [3]:
dfp = results.reset_index()
fig = px.bar(dfp.melt(id_vars="Model", value_vars=["RMSE", "MAE"]),
             x="Model", y="value", color="variable", barmode="group",
             title="RMSE and MAE by Model", labels={"value": "Error", "variable": "Metric"})
fig.update_layout(xaxis_tickangle=-30)
save_fig(fig, "08_rmse_mae")

## F1@10 by model

In [4]:
fig = px.bar(dfp.sort_values("F1@10", ascending=False), x="Model", y="F1@10",
             title="F1@10 by Model", color="F1@10", color_continuous_scale="Teal", text_auto=".3f")
fig.update_layout(coloraxis_showscale=False, xaxis_tickangle=-30)
save_fig(fig, "09_f1_at_10")

## Takeaway

Lead with the Stacked Hybrid's best RMSE/MAE; note Popularity's high F1@10 is a sampled-negatives artifact (it is worst on RMSE). The hybrids beat every CB-only and CF-only baseline on rating accuracy.